# Lab | RAG from Scratch
## A privacy assistant for the GDPR

The bank from this week's labs has a data protection team, and it gets the same questions every day, from customers and from colleagues: *Can I ask you to delete my data? How long do we have to report a data breach? Do we need consent for this?* The answers are all in one document, the **General Data Protection Regulation (GDPR)**, Europe's data protection law, but it is 88 pages long.

Your job is to build a first version of a **privacy assistant**: it answers questions from the GDPR, cites the passages it used so the team can check them, and says so when the GDPR does not answer the question. Then you **evaluate** it on 10 questions with known answers, to decide whether it is good enough to show the team.

**This lab practises** retrieval-augmented generation. You will use, from today's **RAG from Scratch** lesson: loading a PDF, chunking with overlap and metadata, embeddings with MiniLM, a Chroma collection, retrieval with metadata filters, a grounded prompt with citations and refusals, and the hit rate and citation checks.

**By the end, you can:**

1. Implement a RAG pipeline (chunk, embed, store, retrieve, generate) over a new document.
2. Evaluate it with a test set, and explain whether each failure comes from retrieval or from generation.

Each challenge ends with a **checkpoint** that tells you what you should see. Retrieval gives the same results every time; the LLM's answers change a little from run to run. **The whole lab costs about one US cent** with the model below; the last cell shows your total.

**The document** is `data/gdpr.pdf`, [Regulation (EU) 2016/679](https://eur-lex.europa.eu/eli/reg/2016/679/oj) as published in the Official Journal, from EUR-Lex (© European Union, https://eur-lex.europa.eu, reused under [Commission Decision 2011/833/EU](https://eur-lex.europa.eu/eli/dec/2011/833/oj)). Nothing your assistant says is legal advice.

## Setup

You need `pip install pypdf sentence-transformers chromadb openai python-dotenv`. As in yesterday's lab, create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...`. The repository's `.gitignore` already lists `.env`: **never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries, the OpenAI client, a `cost` function and a `track` function that logs the tokens of every call, as yesterday. It also gives you `load_pdf`, the loader from the lesson, with the noise pattern changed for this document: the GDPR prints a footer such as `4.5.2016 L 119/41 Official Journal of the European Union EN` on every page.

In [1]:
import os
import re
import logging
import warnings
from collections import Counter

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 90)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []

def cost(input_tokens, output_tokens):
    price_in, price_out = PRICES[MODEL]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

NOISE = re.compile(r"L 119/\d+ Official Journal of the European Union")

def load_pdf(path):
    """Return one dictionary per page, {"page": number, "text": text}, without the page footers."""
    pages = []
    for number, page in enumerate(PdfReader(path).pages, start=1):
        lines = [line.strip() for line in page.extract_text().split("\n")]
        lines = [line for line in lines if line and not NOISE.search(line)]
        pages.append({"page": number, "text": "\n".join(lines)})
    return pages

Key found: True


**Checkpoint:** `Key found: True`, and no error.

## Challenge 1: load and chunk the GDPR

1. Load `../data/gdpr.pdf` with `load_pdf`, and print the number of pages and the total number of words.
2. Find the page where **Article 17** (*Right to erasure*) starts, and print the first 600 characters of that page. Look at the text: is it clean enough to use?
3. Write `chunk_document(pages, size=150, overlap=30)`. It returns a list of chunks, each a dictionary with an `id` (`"gdpr-0"`, `"gdpr-1"`, ...), the `text`, the `page` where the chunk starts and its `section`: `"Recitals"` before Article 1, then `"Article 1"`, `"Article 2"`, and so on (the section most of the chunk's words come from). In the GDPR, every article starts with a line such as `Article 17`.
4. Chunk the document and print the number of chunks and the share that are recitals.
5. Load the embedding model with `embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")`, count the tokens of every chunk with `embedder.tokenizer`, and print the median and the share of chunks over the model's 256-token limit.

In [2]:
# 1. Load
pages = load_pdf("../data/gdpr.pdf")
words_per_page = [len(p["text"].split()) for p in pages]
print(f"{len(pages)} pages, {sum(words_per_page):,} words")

# 2. The page of Article 17
article_17 = next(p for p in pages if "Article 17" in p["text"].split("\n"))
start = article_17["text"].index("\nArticle 17\n") + 1            # Articles 15 and 16 come first on this page
print(f"\nArticle 17 starts on page {article_17['page']}:\n")
print(article_17["text"][start:start + 600])

# 3. Chunking function
HEADING = re.compile(r"^(Article \d+)$")

def chunk_document(pages, size=150, overlap=30):
    """Split the pages into chunks of `size` words that overlap by `overlap` words, with page and section."""
    words = []                                   # (word, page, section) for every word of the document
    section = "Recitals"                         # everything before Article 1
    for page in pages:
        for line in page["text"].split("\n"):
            match = HEADING.match(line)
            if match:
                section = match.group(1)
            for word in line.split():
                words.append((word, page["page"], section))

    chunks = []
    step = size - overlap
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        chunks.append({
            "id": f"gdpr-{len(chunks)}",
            "text": " ".join(word for word, _, _ in piece),
            "page": piece[0][1],
            "section": Counter(section for _, _, section in piece).most_common(1)[0][0],
        })
        if start + size >= len(words):
            break
    return chunks

# 4. Chunks and recitals
chunks = chunk_document(pages)
sections = pd.Series([c["section"] for c in chunks])
print(f"\n{len(chunks)} chunks, {(sections == 'Recitals').mean():.0%} of them recitals")

# 5. Tokens per chunk
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
tokens = np.array([len(embedder.tokenizer(c["text"])["input_ids"]) for c in chunks])
print(f"Median tokens per chunk: {np.median(tokens):.0f}, "
      f"over the {embedder.max_seq_length}-token limit: {np.mean(tokens > embedder.max_seq_length):.0%}")

88 pages, 54,932 words

Article 17 starts on page 43:

Article 17
Right to erasure (‘right to be forgotten’)
1. The data subject shall have the right to obtain from the controller the erasure of personal data concerning him or
her without undue delay and the controller shall have the obligation to erase personal data without undue delay where
one of the following grounds applies:
(a)  the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise
processed;

458 chunks, 45% of them recitals


Median tokens per chunk: 174, over the 256-token limit: 0%


The text is clean enough to use. Page 43 actually starts with Article 15 and 16, Article 17 comes in the middle of the page, but the heading, the title *Right to erasure ('right to be forgotten')* and the numbered paragraphs come out in the right order and the footer is gone. The only small things are the curly quotes and some double spaces after the `(a)`, `(b)` points, but `split()` takes care of the spaces anyway.

**Checkpoint:** 88 pages and about 55,000 words. Article 17 starts on page 43, and the text is clean: the heading, the title *Right to erasure ('right to be forgotten')* and the numbered paragraphs, with no footer. About 458 chunks, of which about 45% are recitals (the GDPR has 173 recitals for 99 articles). A median of about 174 tokens per chunk and none over 256: 150-word chunks fit MiniLM.

## Challenge 2: index the chunks

1. Embed the text of every chunk with `embedder`, normalised, and print the shape of the result.
2. Create a Chroma collection called `"gdpr"` (in memory, with `chromadb.Client()`) that compares vectors with cosine distance, and add the ids, texts, embeddings and metadata (`page` and `section`) of all chunks. Print `collection.count()`.

In [3]:
# 1. Embeddings
embeddings = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)
print(embeddings.shape)

# 2. Collection
chroma = chromadb.Client()

def make_collection(name, chunks, embeddings):
    """Create a cosine-distance Chroma collection with the texts, embeddings and metadata of the chunks."""
    collection = chroma.create_collection(name, configuration={"hnsw": {"space": "cosine"}})
    collection.add(
        ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        embeddings=embeddings,
        metadatas=[{"page": c["page"], "section": c["section"]} for c in chunks],
    )
    return collection

collection = make_collection("gdpr", chunks, embeddings)
print(collection.count(), "chunks in the collection")

(458, 384)
458 chunks in the collection


**Checkpoint:** `(458, 384)` and `458 chunks in the collection` (the same number as your chunks, whatever it is).

## Challenge 3: retrieve

1. Write `retrieve(question, k=5, where=None)`: it embeds the question with the same model, queries the collection, and returns a DataFrame with the columns `id`, `section`, `page`, `distance` and `text`.
2. Define `RULES_ONLY`, a `where` filter that excludes the recitals.
3. Retrieve the top 5 chunks for *Can I ask a company to delete my data?*, first without a filter and then with `RULES_ONLY`.

In [4]:
# 1. Retrieval function
def retrieve(question, k=5, where=None, collection=collection):
    """Return the k chunks closest to the question as a DataFrame."""
    query = embedder.encode([question], normalize_embeddings=True)
    result = collection.query(query_embeddings=query, n_results=k, where=where)
    return pd.DataFrame({
        "id": result["ids"][0],
        "section": [m["section"] for m in result["metadatas"][0]],
        "page": [m["page"] for m in result["metadatas"][0]],
        "distance": np.round(result["distances"][0], 3),
        "text": result["documents"][0],
    })

# 2. Filter
RULES_ONLY = {"section": {"$ne": "Recitals"}}

# 3. Without and with the filter
question = "Can I ask a company to delete my data?"
print("Without a filter:")
display(retrieve(question))
print("With RULES_ONLY:")
display(retrieve(question, where=RULES_ONLY))

Without a filter:


,id,section,page,distance,text
0,gdpr-255,Article 15,43,0.508,right to request from the controller rectification or erasure of personal data or rest...
1,gdpr-186,Recitals,29,0.512,"data in the employment context, in particular for the conditions under which personal ..."
2,gdpr-76,Recitals,12,0.514,in particular in the context of online services and online identifiers. A controller s...
3,gdpr-77,Recitals,12,0.533,has withdrawn his or her consent or objects to the processing of personal data concern...
4,gdpr-78,Recitals,13,0.536,"freedom of expression and information, for compliance with a legal obligation, for the..."


With RULES_ONLY:


,id,section,page,distance,text
0,gdpr-255,Article 15,43,0.508,right to request from the controller rectification or erasure of personal data or rest...
1,gdpr-246,Article 13,41,0.538,"Article 6(1) or point (a) of Article 9(2), the existence of the right to withdraw cons..."
2,gdpr-257,Article 17,43,0.541,right to obtain from the controller without undue delay the rectification of inaccurat...
3,gdpr-220,Article 5,35,0.542,"are processed, are erased or rectified without delay (‘accuracy’); (1) Directive (EU) ..."
4,gdpr-444,Article 88,84,0.548,of the rights and freedoms in respect of the processing of employees' personal data in...


**Checkpoint:** Without the filter, the first chunk comes from Article 15 and the other four are recitals. With `RULES_ONLY`, all five are articles, and Article 17 (page 43) is third.

**Question:** which article answers the question, and in which position does it appear with and without the filter? Why do the recitals compete with the articles?

The question is answered by **Article 17** (*Right to erasure*). Without the filter it's not in the top 5 at all: the first chunk is from Article 15 (it lists the right to request erasure as one of the things a company has to tell you about) and the other four are recitals. With `RULES_ONLY` Article 17 shows up in **third** place (gdpr-257, page 43, distance 0.541), behind Article 15 and Article 13.

The recitals compete because they explain the articles, often with the same or even plainer words ("erasure", "withdrawn his or her consent", "online services"). So for the embedding they are just as close to the question as the rule itself. And there are a lot of them: 45% of all chunks are recitals, so they crowd out the articles. The distances are also very close together (0.508 to 0.548), so small differences in wording decide the order.

## Challenge 4: grounded answers

1. Write the system prompt of the privacy assistant. It must answer **only** from the excerpts, cite the excerpt ids in square brackets after each statement (for example `[gdpr-12]`), reply exactly `I don't know based on the GDPR.` and nothing else when the excerpts do not contain the answer, and answer in 2 to 4 sentences of plain English.
2. Write `answer(question, k=5, where=RULES_ONLY)`. It retrieves the chunks, builds the context (each chunk labelled with its id, section and page, as in the lesson), calls the model with the system prompt and a user message containing the excerpts and the question, tracks the call as `"answers"`, and returns a dictionary with the `answer`, the `cited` ids (find them in the answer with a regular expression), the `retrieved` ids and their `sections`.
3. Print the answers to these three questions, each with the sections it cited:
   - *How long does a company have to report a data breach to the authority?*
   - *Can I get a copy of the personal data a company holds about me?*
   - *What is the VAT rate for online services in Spain?*

In [5]:
# 1. System prompt
SYSTEM = """You are a privacy assistant that answers questions about the GDPR using ONLY the excerpts provided.
Rules:
- Base every statement on the excerpts. Do not use outside knowledge.
- After each statement, cite the excerpt id(s) in square brackets, e.g. [gdpr-12].
- If the excerpts do not contain the answer, reply exactly "I don't know based on the GDPR." and nothing else.
- Answer in 2-4 sentences of plain English."""

REFUSAL = "I don't know based on the GDPR."

# 2. Answer function
def build_context(hits):
    """Label every retrieved chunk with its id, section and page."""
    return "\n\n".join(f"[{row.id}] ({row.section}, page {row.page})\n{row.text}" for row in hits.itertuples())

def answer(question, k=5, where=RULES_ONLY, collection=collection):
    """Retrieve the chunks, ask the model with the grounded prompt and return the answer with its citations."""
    hits = retrieve(question, k=k, where=where, collection=collection)
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Excerpts:\n{build_context(hits)}\n\nQuestion: {question}"},
    ]
    response = track(client.chat.completions.create(model=MODEL, messages=messages), "answers")
    text = response.choices[0].message.content
    cited = re.findall(r"gdpr-\d+", text)
    return {"answer": text, "cited": sorted(set(cited), key=lambda c: int(c.split("-")[1])),
            "retrieved": list(hits["id"]), "sections": list(hits["section"])}

# 3. Three questions
section_of = {c["id"]: c["section"] for c in chunks}
page_of = {c["id"]: c["page"] for c in chunks}

for question in ["How long does a company have to report a data breach to the authority?",
                 "Can I get a copy of the personal data a company holds about me?",
                 "What is the VAT rate for online services in Spain?"]:
    result = answer(question)
    print(question)
    print(" ->", result["answer"])
    print("    cited:", [(c, section_of.get(c, "unknown id"), f"page {page_of.get(c)}") for c in result["cited"]], "\n")

How long does a company have to report a data breach to the authority?
 -> A company (the controller) must notify the relevant supervisory authority **without undue delay**, and **where feasible, not later than 72 hours after becoming aware of the personal data breach**. [gdpr-294]  

If it cannot be done within 72 hours, the notification must include **reasons for the delay**. [gdpr-294]
    cited: [('gdpr-294', 'Article 33', 'page 52')] 



Can I get a copy of the personal data a company holds about me?
 -> Yes. You have the right to request that the controller provides you with a copy of the personal data undergoing processing. [gdpr-256]  

If you request additional copies, the controller may charge a reasonable fee based on administrative costs, and if you request electronically, they must provide the information in a commonly used electronic form unless you ask otherwise. [gdpr-256]
    cited: [('gdpr-256', 'Article 15', 'page 43')] 



What is the VAT rate for online services in Spain?
 -> I don't know based on the GDPR. [gdpr-453] [gdpr-454] [gdpr-456] [gdpr-427] [gdpr-436]
    cited: [('gdpr-427', 'Article 79', 'page 80'), ('gdpr-436', 'Article 83', 'page 82'), ('gdpr-453', 'Article 94', 'page 86'), ('gdpr-454', 'Article 97', 'page 86'), ('gdpr-456', 'Article 98', 'page 87')] 



**Checkpoint:** The breach answer says *without undue delay* and, where feasible, *not later than 72 hours* after the company becomes aware of the breach, citing a chunk of Article 33. The copy answer says yes, citing Article 15 (and mentions that further copies can cost a reasonable fee). The VAT question gets `I don't know based on the GDPR.` (sometimes followed by chunk ids, even though the rule says *nothing else*). Your wording will differ.

**Question:** open `data/gdpr.pdf` at the page of one chunk your assistant cited, and compare. Is the answer faithful to the text? Did it leave out anything important?

I checked the breach answer against **Article 33 on page 52** (chunk gdpr-294). It is faithful: *without undue delay*, *where feasible, not later than 72 hours after having become aware of it* and *accompanied by reasons for the delay* are almost word for word from paragraph 1. Nothing was made up.

But it leaves out one important part of the same sentence: the notification is **not needed when the breach is unlikely to result in a risk to the rights and freedoms of natural persons**. So the answer sounds like every breach has to be reported, which is not what the GDPR says. It also doesn't mention that a processor has to notify the controller (paragraph 2) or what the notification has to contain (paragraph 3). For a 2 to 4 sentence answer that is probably okay, but the exception is something the team would really need to know.

The copy answer (Article 15, page 43) is faithful too, but it misses paragraph 4: the right to a copy *shall not adversely affect the rights and freedoms of others*. And the VAT question was refused correctly, but in this run the model put five chunk ids behind the refusal, even though the prompt says *nothing else*.

## Challenge 5: evaluate the assistant

The data protection team wrote 10 questions and, for each one, the article that answers it. Use them to decide whether the assistant is ready to show.

1. Write `hit_rate(k=5, where=None)`: the share of test questions for which at least one of the top k chunks comes from the expected article. Print a table with the hit rate for `k` = 3 and 5, without a filter and with `RULES_ONLY`, and the questions missed in each case.
2. Run `answer` on the 10 questions and build a DataFrame with one row per question and four `True`/`False` columns: `right article retrieved`, `cites` (at least one id), `cites only what it saw` (every cited id was retrieved) and `cites the right article`. Print the mean of each column and show the rows that failed any check.
3. Run `answer` on the two out-of-scope questions in `OUT_OF_SCOPE` and count how many were refused.

In [6]:
TEST_SET = pd.DataFrame([
    ("Can I ask a company to delete my data?", "Article 17"),
    ("How long does a company have to tell the authority about a data breach?", "Article 33"),
    ("What are the legal grounds for processing personal data?", "Article 6"),
    ("How much can a company be fined under the GDPR?", "Article 83"),
    ("Can I get a copy of the personal data a company holds about me?", "Article 15"),
    ("At what age can a child consent to online services on their own?", "Article 8"),
    ("When does a company need a data protection officer?", "Article 37"),
    ("Can I move my data from one service provider to another?", "Article 20"),
    ("What counts as valid consent?", "Article 7"),
    ("Can a decision about me be made only by an algorithm?", "Article 22"),
], columns=["question", "expected"])
OUT_OF_SCOPE = ["What is the VAT rate for online services in Spain?",
                "How many vacation days do employees get in France?"]

# 1. Hit rate
def hit_rate(k=5, where=None, collection=collection):
    """Share of test questions with at least one top-k chunk from the expected article, and the missed questions."""
    queries = embedder.encode(list(TEST_SET["question"]), normalize_embeddings=True)
    result = collection.query(query_embeddings=queries, n_results=k, where=where)
    hits = [expected in [m["section"] for m in metas]
            for expected, metas in zip(TEST_SET["expected"], result["metadatas"])]
    return np.mean(hits), [q for q, hit in zip(TEST_SET["question"], hits) if not hit]

rows = []
for k in [3, 5]:
    for name, where in [("none", None), ("rules only", RULES_ONLY)]:
        rate, missed = hit_rate(k=k, where=where)
        rows.append({"k": k, "filter": name, "hit rate": round(rate, 2), "missed": missed})
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows))
pd.set_option("display.max_colwidth", 90)

# 2. Answer checks
rows = []
for question, expected in TEST_SET.itertuples(index=False):
    result = answer(question)
    rows.append({
        "question": question,
        "expected": expected,
        "right article retrieved": expected in result["sections"],
        "cites": len(result["cited"]) > 0,
        "cites only what it saw": set(result["cited"]) <= set(result["retrieved"]),
        "cites the right article": expected in [section_of.get(c) for c in result["cited"]],
        "cited sections": sorted({section_of.get(c, "unknown id") for c in result["cited"]}),
        "answer": result["answer"],
    })
checks = pd.DataFrame(rows)
CHECKS = ["right article retrieved", "cites", "cites only what it saw", "cites the right article"]
print(checks[CHECKS].mean().round(2))
failed = checks[~checks[CHECKS].all(axis=1)]
display(failed.drop(columns="answer"))
for row in failed.itertuples():
    print(f"{row.question} (expected {row.expected})\n -> {row.answer}\n")

# 3. Out-of-scope questions
refused = 0
for question in OUT_OF_SCOPE:
    result = answer(question)
    refused += result["answer"].strip().startswith(REFUSAL)
    print(question, "\n ->", result["answer"], "\n")
print(f"{refused} of {len(OUT_OF_SCOPE)} out-of-scope questions refused")

,k,filter,hit rate,missed
0,3,none,0.5,"[Can I ask a company to delete my data?, How long does a company have to tell the authority about a data breach?, What are the legal grounds for processing personal data?, Can I move my data from one service provider to another?, Can a decision about me be made only by an algorithm?]"
1,3,rules only,0.8,"[What are the legal grounds for processing personal data?, Can I move my data from one service provider to another?]"
2,5,none,0.6,"[Can I ask a company to delete my data?, What are the legal grounds for processing personal data?, Can I move my data from one service provider to another?, Can a decision about me be made only by an algorithm?]"
3,5,rules only,0.8,"[What are the legal grounds for processing personal data?, Can I move my data from one service provider to another?]"


right article retrieved    0.8
cites                      0.9
cites only what it saw     1.0
cites the right article    0.8
dtype: float64


,question,expected,right article retrieved,cites,cites only what it saw,cites the right article,cited sections
2,What are the legal grounds for processing personal data?,Article 6,False,True,True,False,"[Article 13, Article 17]"
7,Can I move my data from one service provider to another?,Article 20,False,False,True,False,[]


What are the legal grounds for processing personal data? (expected Article 6)
 -> The excerpts mention that processing can be lawful based on **consent**, including the right to withdraw consent at any time without affecting the lawfulness of processing based on consent before withdrawal. [gdpr-246] They also refer to processing having a “legal ground” (including where consent is withdrawn and there is no other legal ground). [gdpr-258] However, the excerpts provided do not list the full set of all legal grounds for processing personal data. [gdpr-246]

Can I move my data from one service provider to another? (expected Article 20)
 -> I don’t know based on the GDPR.



What is the VAT rate for online services in Spain? 
 -> I don't know based on the GDPR. 



How many vacation days do employees get in France? 
 -> I don't know based on the GDPR. 

2 of 2 out-of-scope questions refused


**Checkpoint:** Hit rates of 0.5 and 0.8 at k = 3, and 0.6 and 0.8 at k = 5 (without and with the filter). With the filter, the two misses are the legal grounds (Article 6) and data portability (Article 20). In the answer checks, about 0.8 of the answers cite the right article and none cites an id it did not see, and the failures are the same two questions, both with `right article retrieved` = `False`. Both out-of-scope questions are refused. The retrieval numbers should match exactly; the answers can differ a little.

**Question:** for each failure, say whether it comes from retrieval or from generation, and why. Then decide: would you show this assistant to the data protection team? What would you change first, and what would you tell them about how to use it?

Both failures come from **retrieval**. In both rows `right article retrieved` is `False`, so the model never saw the article it was supposed to cite.

1. **Legal grounds (Article 6).** Article 6 doesn't use the words "legal grounds". It is called *Lawfulness of processing* and says *processing shall be lawful only if...*. Other articles (13, 17, 21) talk about consent, "legal ground" or "legitimate grounds", so they win. When I looked a bit further, the closest Article 6 chunk was at rank 6, just outside k = 5, and it's from paragraph 4, not the list of the six grounds. The model then answered from Article 13 and 17 and said that processing can be based on consent. It even says that the excerpts don't list all the grounds, which is honest, but a user who only reads the first sentence gets an incomplete answer that looks checked because it has citations.
2. **Data portability (Article 20).** The user says *move my data from one service provider to another*, the GDPR says *right to data portability* and *transmit those data to another controller*. "Move data" is closer to the chapter on *transfers* to third countries (Article 44), so that's what came back, and all distances were high (0.63 and more). The closest Article 20 chunk was only at rank 15. In this run the model refused, which is the right thing to do with that context. In my earlier runs it gave "I don't know" with five chunk ids behind it once and once it actually answered about international transfers, citing Articles 15 and 44, which is just wrong.

I didn't find a real generation failure: whenever the right article was retrieved (8 of 10), the answer cited it, and the assistant never cited an id it hadn't seen. The `cites` mean of 0.9 is only because of the portability refusal, where not citing is correct. One thing I noticed though: that refusal came with a curly apostrophe (`don’t`), so an exact string check for the refusal sentence would have missed it.

**Would I show it to the team?** Yes, but as an internal research aid for the data protection team, not as something that answers customers directly. 8 of 10 is a good start, but the two misses are basic questions that the team gets all the time, and the wrong-article answers look convincing because they have citations.

**What I would change first:** retrieval, because that's where both failures come from.
- Put the article title in front of every chunk (e.g. "Article 20, Right to data portability"), so the plain-language topic is in every chunk of that article and not only in the first one.
- Try a stronger embedding model like `text-embedding-3-small` from the lesson, or combine the embeddings with a keyword search (BM25), so "portability" or "lawfulness" can match directly.
- Write more test questions in the way customers actually ask them, 10 questions is too few to tell small improvements apart.

**What I would tell the team:**
- Always open the cited article and check the answer, especially exceptions and conditions, which the short answers tend to leave out (like the "unlikely to result in a risk" part of Article 33).
- "I don't know based on the GDPR" means the assistant didn't find it, not that the GDPR doesn't cover it. Asking again with the legal term ("data portability", "lawfulness of processing") often helps.
- It only knows the GDPR text, not national laws or guidelines, and it is not legal advice.
- Don't paste customer names or other personal data into the questions, since every question and the retrieved chunks are sent to OpenAI.

## What did it cost?

Run the cell below to see the tokens and the cost of the lab.

In [7]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

Total: 15 calls, $0.0045


,input,output,calls,cost ($)
step,,,,
answers,16626,923,15,0.00448


**Checkpoint:** 15 calls and about half a US cent. Indexing with MiniLM was free; every answer costs one call of about 1,100 input tokens.

## Extra: tune the chunk size, or use your own document

Choose one:

+ **Chunk size.** Build one collection per chunk size (100, 150, 200 and 400 words, with an overlap of a fifth of the size) and compare their hit rate at k = 5 with `RULES_ONLY`. Which size works best on the GDPR, and is it the same as for the AI Act in the lesson? Be careful: with only 10 questions, a difference of one question is 0.1.
+ **Your own document.** Index a PDF you are allowed to use (your company's public privacy policy, a manual, a public report), write five questions with the page that answers them, and measure the hit rate. What did you have to change in `load_pdf` and `chunk_document`?

In [8]:
# Chunk size: one collection per size, overlap of a fifth of the size, hit rate at k = 5 with RULES_ONLY
rows = []
for size in [100, 150, 200, 400]:
    sized_chunks = chunk_document(pages, size=size, overlap=size // 5)
    sized_embeddings = embedder.encode([c["text"] for c in sized_chunks], normalize_embeddings=True,
                                       batch_size=64, show_progress_bar=False)
    sized_collection = make_collection(f"gdpr_{size}", sized_chunks, sized_embeddings)
    sized_tokens = np.array([len(embedder.tokenizer(c["text"])["input_ids"]) for c in sized_chunks])
    rate, missed = hit_rate(k=5, where=RULES_ONLY, collection=sized_collection)
    rows.append({"words per chunk": size, "chunks": len(sized_chunks),
                 "share over 256 tokens": f"{np.mean(sized_tokens > 256):.0%}",
                 "hit rate (k=5, rules only)": round(rate, 2),
                 "missed": [TEST_SET.set_index("question").loc[q, "expected"] for q in missed]})
pd.DataFrame(rows)

,words per chunk,chunks,share over 256 tokens,"hit rate (k=5, rules only)",missed
0,100,687,0%,0.8,"[Article 6, Article 20]"
1,150,458,0%,0.8,"[Article 6, Article 20]"
2,200,344,5%,0.8,"[Article 6, Article 20]"
3,400,172,99%,0.7,"[Article 6, Article 8, Article 20]"


**Conclusions:** I chose the chunk size option. With `RULES_ONLY` and k = 5, 100, 150 and 200 words all get the same hit rate of 0.8 and miss the same two questions (Article 6 and 20). Only 400 words is worse with 0.7, it also misses the child consent question (Article 8). That makes sense, because 99% of the 400-word chunks are over MiniLM's 256-token limit, so the model only embeds the first part of every chunk.

So on the GDPR there is no single best size between 100 and 200, and with 10 questions one question is already 0.1, so I can't really tell them apart. In the lesson on the AI Act 100 and 150 words did best and longer chunks got worse. It's roughly the same pattern here, only that 200 words still holds up on the GDPR. I would stay with 150 words: it's as good as the others, no chunk goes over the token limit (at 200 words already 5% do), and the chunks are long enough to hold a whole paragraph of a rule.

The more important finding is that the two misses stay the same for every chunk size. So the chunk size is not the problem, the problem is that the users' words ("legal grounds", "move my data") are different from the words of the law. That needs the fixes from Challenge 5 (titles in the chunks, a better embedding model or keyword search), not different chunks.